# 02 — Model Training & Fine-Tuning Pipeline

**Rubric Alignment:**
- **Criterion 2 (20%) — Machine Learning Methods:** Custom Fusion-ViT architecture with gated multimodal fusion, theoretical justification, pros and cons.
- **Criterion 4 (20%) — Benchmark Methods:** At least 3+ benchmark models (ViT-B/16, Swin-T, ResNet-50, MobileNetV3-Large), architectural selection rationale and novelty.
- **Criterion 5 (20%) — Implementation:** From-scratch Vision Transformer (no `nn.MultiheadAttention` shortcuts), modular training loop, step-by-step comments, memory/complexity awareness.

**Prerequisites:** Run `01_eda.ipynb` first so that `outputs/splits.csv` and `outputs/features_33d.npy` exist.

In [1]:
# ── Imports & Environment Setup ──────────────────────────────────────────────
import sys, os, time, json
sys.path.insert(0, '../modules')  # Load data, features, models modules

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

import data as D
import models as M

# Auto-detect device hierarchy: CUDA > MPS (Apple Silicon GPU) > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Selected execution device: {DEVICE}')

# Ensure output directories exist
RESULTS_DIR = '../outputs/results'
CKPT_DIR = os.path.join(RESULTS_DIR, 'checkpoints')
PREDS_DIR = os.path.join(RESULTS_DIR, 'predictions')
CURVES_DIR = os.path.join(RESULTS_DIR, 'training_curves')
METRICS_DIR = os.path.join(RESULTS_DIR, 'metrics')

for d in [CKPT_DIR, PREDS_DIR, CURVES_DIR, METRICS_DIR]:
    os.makedirs(d, exist_ok=True)

print('All target directories initialized.')

# Verify canonical split & feature provenance manifest
SPLIT_MANIFEST = D.save_or_verify_manifest(
    splits_csv='../outputs/splits.csv',
    features_npy='../outputs/features_33d.npy',
    manifest_path='../outputs/split_manifest.json',
    strict=True,
)
print(f"Split Manifest OK: {SPLIT_MANIFEST['split_version']} | splits_sha256={SPLIT_MANIFEST['splits_sha256'][:12]}...")

Selected execution device: mps
All target directories initialized.
Split Manifest OK: v1_stratified_seed42 | splits_sha256=7d3bc86cca81...


---
## 1. Architectural Paradigms & Benchmark Selection Rationale

| ID | Model Architecture | Paradigm & Mechanism | Why We Selected It (Selection Rationale & Novelty) | Pros & Cons |
| :--- | :--- | :--- | :--- | :--- |
| **M1** | **ViT-Scratch** | Pure self-attention on $4 \times 4$ patches ($N=60$ tokens), 6 layers, 192 dim, 3 heads | Implemented strictly from raw PyTorch linear ops to demonstrate first-principles mastery of the Transformer encoder without library shortcuts. | **Pros:** Native $40 \times 24$ input, zero upsampling distortion, full global context from layer 1.<br>**Cons:** Lacks large-scale pretraining; relies on aggressive regularization. |
| **M2** | **Fusion-ViT (Main Method)** | Dual-branch multimodal architecture: ViT image stream + MLP thermal feature stream fused via learnable gating $g = \sigma(W [h_{img}; h_{feat}])$ | **Novelty:** Unifies spatial infrared pixel patterns with 33 engineered radiometric/thermal invariants, dynamically modulating the contribution of each modality per sample. | **Pros:** Exploits both image context and exact radiometric statistics; robust to local thermal noise.<br>**Cons:** Requires paired feature extraction preprocessing. |
| **M3** | **ViT-B/16 (Pretrained)** | Standard Vision Transformer ($16 \times 16$ patches, 86M params) pretrained on ImageNet-1k | Proposal's anchor model. Tests whether massive large-scale pretraining on natural images transfers effectively to low-resolution infrared thermal heatmaps. | **Pros:** Exceptional transfer representations, rich attention maps.<br>**Cons:** Severe $40 \times 24 \to 224 \times 224$ upsampling artifacts, high compute latency. |
| **B1** | **Swin-T (Baseline 1)** | Hierarchical Vision Transformer with Shifted Windows ($O(N)$ linear complexity) | Tests whether local windowed self-attention and cross-window shifting better capture sharp fault boundaries (e.g., diode strings, cell cracks) than global patch ViT. | **Pros:** Multi-scale feature maps, computationally efficient window attention.<br>**Cons:** Shifted window partitioning can miss whole-panel global thermal drift. |
| **B2** | **ResNet-50 (Baseline 2)** | Deep Residual CNN (25M params) with $3 \times 3$ convolutions and identity skip connections | The established industry benchmark in photovoltaic defect detection literature; acts as our convolutional control baseline. | **Pros:** Strong spatial inductive bias, fast training convergence, well-calibrated.<br>**Cons:** Local receptive field struggles with global module-wide thermal balance. |
| **B3** | **MobileNetV3-Large (Baseline 3)** | Lightweight CNN (4.2M params) using Depthwise Separable Convolutions & Squeeze-and-Excitation | Edge-computing benchmark. Evaluates whether lightweight models can achieve deployable accuracy directly on UAV embedded processors. | **Pros:** Lowest parameter count, minimal memory bandwidth, ultra-high FPS.<br>**Cons:** Lower capacity on fine-grained rare categories (e.g., Diode-Multi). |

---
## 2. Loss Function & Optimization Strategy

1. **Imbalance-Weighted Cross-Entropy:**
   Class frequencies range from 10,000 (No-Anomaly) to 175 (Diode-Multi). We apply inverse square-root frequency weighting:
   $$w_c = \frac{1}{\sqrt{N_c + \epsilon}}, \quad \tilde{w}_c = \frac{w_c}{\frac{1}{C}\sum_{k=1}^C w_k}$$
2. **Label Smoothing ($\alpha = 0.1$):**
   Replaces hard one-hot targets with $y_{smooth} = (1 - \alpha) y + \frac{\alpha}{C}$, preventing the network from becoming overconfident on ambiguous thermal boundaries.
3. **AdamW Optimizer + Cosine Annealing:**
   Decoupled weight decay (0.05 for scratch, 0.01 for pretrained) with cosine learning rate decay down to $10^{-6}$.

In [2]:
# ── Reproducibility Seed Utility ──────────────────────────────────────────────
def set_seed(seed: int = 42):
    """Fix random seeds across python, numpy, and PyTorch for exact reproducibility."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f'Random seed locked to {seed}')

---
## 3. Generic Model Trainer & Evaluator Engine

Encapsulates forward pass (handling both single-input vision models and dual-input multimodal fusion), metric tracking, early stopping on validation macro-F1, checkpoint serialization, and test evaluation. Also defines `run_vitb16_fast` for M3 (`ViT-B/16`) cached-token partial fine-tuning so all runner functions are available before Sections 4–6.

In [3]:
def train_one_epoch(model, loader, criterion, optimizer, device, is_fusion=False):
    """Execute a single training epoch with batch iteration."""
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for imgs, feats, targets in loader:
        imgs, feats, targets = imgs.to(device), feats.to(device), targets.to(device)
        optimizer.zero_grad()

        # Multimodal dispatch: FusionViT receives both image and 33-dim feature vector
        if is_fusion:
            logits = model(imgs, feats)
        else:
            logits = model(imgs)

        loss = criterion(logits, targets)
        loss.backward()
        # Gradient clipping to stabilize transformer attention gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return epoch_loss, epoch_acc, epoch_macro_f1


@torch.no_grad()
def evaluate_dataset(model, loader, criterion, device, is_fusion=False):
    """Evaluate model performance on val or test split."""
    model.eval()
    running_loss = 0.0
    all_preds, all_targets, all_probs = [], [], []

    for imgs, feats, targets in loader:
        imgs, feats, targets = imgs.to(device), feats.to(device), targets.to(device)

        if is_fusion:
            logits = model(imgs, feats)
        else:
            logits = model(imgs)

        loss = criterion(logits, targets)
        running_loss += loss.item() * imgs.size(0)

        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()

        all_probs.append(probs)
        all_preds.extend(preds)
        all_targets.extend(targets.cpu().numpy())

    total_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    all_probs = np.concatenate(all_probs, axis=0)
    return total_loss, acc, macro_f1, np.array(all_preds), all_probs, np.array(all_targets)


def plot_training_history(history, model_name, seed, save_path):
    """Generate and save publication-quality loss and macro-F1 learning curves."""
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    ax1.plot(epochs, history['train_loss'], label='Train Loss', color='#4C72B0', lw=2)
    ax1.plot(epochs, history['val_loss'], label='Val Loss', color='#DD8452', lw=2, linestyle='--')
    ax1.set_title(f'{model_name} (Seed {seed}) — Loss Progression', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Cross-Entropy Loss (smoothed)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history['val_macro_f1'], label='Val Macro-F1', color='#55A868', lw=2)
    ax2.plot(epochs, history['val_acc'], label='Val Accuracy', color='#8172B3', lw=1.5, linestyle=':')
    ax2.set_title(f'{model_name} (Seed {seed}) — Validation Metric Curves', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Score [0-1]')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.close()

In [4]:
def load_existing_metrics(model_name: str, seed: int, skip_existing: bool = True):
    """Return saved test metrics only if trained on the active split manifest."""
    if not skip_existing:
        return None
    metrics_path = os.path.join(METRICS_DIR, f'{model_name}_seed{seed}_metrics.json')
    ckpt_path    = os.path.join(CKPT_DIR, f'{model_name}_seed{seed}_best.pt')
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            data = json.load(f)
        rec_sha = data.get('splits_sha256')
        if rec_sha is not None and rec_sha != SPLIT_MANIFEST['splits_sha256']:
            print(f'Stale metrics for {model_name} (seed {seed}): split checksum mismatch — retraining.')
            return None
        if not os.path.exists(ckpt_path):
            print(f'Missing checkpoint {ckpt_path} for {model_name} (seed {seed}) — retraining.')
            return None
        print(f'Skipping {model_name} (seed {seed}) — already trained ({metrics_path}).')
        return data
    return None


def run_training_experiment(
    model_name: str,
    seed: int = 0,
    epochs: int = 40,
    patience: int = 8,
    lr: float = 5e-4,
    weight_decay: float = 0.05,
    batch_size: int = 64,
    device = DEVICE,
    skip_existing: bool = True
):
    """Full training execution loop for any registered model.
    
    1. Skips immediately and returns saved metrics if already trained (when skip_existing=True).
    2. Sets random seeds.
    3. Configures resolution (40x24 native for scratch/fusion vs 224x224 for pretrained).
    4. Initializes data loaders, model, loss function, and scheduler.
    5. Executes training with validation early stopping on Macro-F1.
    6. Loads best checkpoint and evaluates once on unseen test set.
    7. Serializes metrics, learning history, checkpoints, and test predictions.
    """
    cached = load_existing_metrics(model_name, seed, skip_existing)
    if cached is not None:
        return cached
    set_seed(seed)
    is_fusion = (model_name == 'fusion_vit')
    is_scratch = model_name in ['vit_scratch', 'fusion_vit']

    # Scratch models take native 40x24 grayscale; pretrained models take 224x224 3-channel
    target_size = (40, 24) if is_scratch else (224, 224)
    in_channels = 1 if is_scratch else 3

    print(f'\n======================================================================')
    print(f'Starting Experiment: {model_name} | Seed: {seed} | Device: {device}')
    print(f'Input Resolution: {target_size} | Channels: {in_channels} | Batch Size: {batch_size}')
    print(f'======================================================================')

    train_loader, val_loader, test_loader, class_weights, _, _ = D.get_dataloaders(
        splits_csv='../outputs/splits.csv',
        features_npy='../outputs/features_33d.npy',
        batch_size=batch_size,
        target_size=target_size,
        in_channels=in_channels,
        num_workers=0
    )

    model = M.get_model(model_name).to(device)
    total_params = M.count_parameters(model)
    print(f'Model instantiated successfully with {total_params:,} trainable parameters.')

    # Label smoothing cross-entropy with inverse square-root class weights
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    history = {'train_loss': [], 'train_acc': [], 'train_macro_f1': [],
               'val_loss': [], 'val_acc': [], 'val_macro_f1': [], 'lr': [], 'epoch_time': []}

    best_val_f1 = -1.0
    epochs_no_improve = 0
    best_ckpt_path = os.path.join(CKPT_DIR, f'{model_name}_seed{seed}_best.pt')

    start_time = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer, device, is_fusion)
        va_loss, va_acc, va_f1, _, _, _ = evaluate_dataset(model, val_loader, criterion, device, is_fusion)
        scheduler.step()
        dt = time.time() - t0

        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['train_macro_f1'].append(tr_f1)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        history['val_macro_f1'].append(va_f1)
        history['lr'].append(current_lr)
        history['epoch_time'].append(dt)

        print(f'Epoch [{epoch:02d}/{epochs:02d}] {dt:.1f}s | Train Loss: {tr_loss:.4f} F1: {tr_f1:.4f} | '
              f'Val Loss: {va_loss:.4f} Acc: {va_acc:.4f} F1: {va_f1:.4f} (Best: {max(best_val_f1, va_f1):.4f})')

        # Checkpoint selection strictly based on validation Macro-F1
        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            epochs_no_improve = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_f1': best_val_f1,
                'model_name': model_name,
                'seed': seed
            }, best_ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'Early stopping triggered after {epoch} epochs (no improvement for {patience} epochs).')
                break

    total_train_time = time.time() - start_time
    print(f'Training complete in {total_train_time/60:.2f} minutes. Best Val Macro-F1: {best_val_f1:.4f}')

    # Save training history and curves
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(CURVES_DIR, f'{model_name}_seed{seed}_history.csv'), index=False)
    plot_training_history(history, model_name, seed, os.path.join(CURVES_DIR, f'{model_name}_seed{seed}_curves.png'))

    # Final Test Set Evaluation using the Best Checkpoint (Evaluated ONCE only)
    print('\nEvaluating best checkpoint on held-out test set...')
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    te_loss, te_acc, te_f1, te_preds, te_probs, te_targets = evaluate_dataset(model, test_loader, criterion, device, is_fusion)

    # Compute per-class F1-scores
    per_class_f1 = f1_score(te_targets, te_preds, average=None, zero_division=0)
    test_results = {
        'model': model_name,
        'seed': seed,
        'split_version': SPLIT_MANIFEST['split_version'],
        'splits_sha256': SPLIT_MANIFEST['splits_sha256'],
        'parameters': total_params,
        'best_val_f1': float(best_val_f1),
        'test_loss': float(te_loss),
        'test_accuracy': float(te_acc),
        'test_macro_f1': float(te_f1),
        'total_training_seconds': float(total_train_time),
        'per_class_f1': {D.CLASS_NAMES[i]: float(per_class_f1[i]) for i in range(len(D.CLASS_NAMES))}
    }

    with open(os.path.join(METRICS_DIR, f'{model_name}_seed{seed}_metrics.json'), 'w') as f:
        json.dump(test_results, f, indent=2)

    np.save(os.path.join(PREDS_DIR, f'{model_name}_seed{seed}_test_preds.npy'), te_preds)
    np.save(os.path.join(PREDS_DIR, f'{model_name}_seed{seed}_test_probs.npy'), te_probs)
    np.save(os.path.join(PREDS_DIR, f'test_targets.npy'), te_targets)

    print(f'Test Set Results for {model_name} (Seed {seed}):')
    print(f'  Test Accuracy : {te_acc:.4f}')
    print(f'  Test Macro-F1 : {te_f1:.4f}')
    print(f'Saved test predictions and metrics successfully.')
    return test_results

In [5]:
import copy, vit_fast as VF


def run_vitb16_fast(seed=0, epochs=20, patience=5, cut=8, input_hw=(160, 96),
                    lr_head=1e-3, lr_blocks=1e-4, weight_decay=0.01,
                    batch_size=128, n_views=2, device=DEVICE, skip_existing=True):
    """Fast partial fine-tuning of ViT-B/16 (M3) on cached frozen-prefix tokens."""
    cached = load_existing_metrics('vitb16', seed, skip_existing)
    if cached is not None:
        return cached
    set_seed(seed)
    VIEWS = [(0, 0), (1, 0), (0, 1), (1, 1)][:n_views]   # flip augmentation views
    df = pd.read_csv('../outputs/splits.csv')
    d = {s: df[df.split == s].reset_index(drop=True) for s in ('train', 'val', 'test')}
    y = {s: torch.tensor(d[s]['label'].values) for s in d}
    print('Loading images...'); u8 = {s: VF.load_uint8(d[s]) for s in d}
    print('Building ViT-B/16 (downloads weights on first use)...')

    m = VF.build_vitb16(input_hw, pretrained=True, num_classes=len(D.CLASS_NAMES))
    print(f'Caching tokens on {device} ...'); t0 = time.time()
    tr_tok = torch.stack([VF.cache_tokens(m, u8['train'], input_hw, cut, device, v) for v in VIEWS])  # (V,N,61,768)
    va_tok = VF.cache_tokens(m, u8['val'],  input_hw, cut, device)
    te_tok = VF.cache_tokens(m, u8['test'], input_hw, cut, device)
    print(f'Token caching: {time.time()-t0:.0f}s | train cache {tuple(tr_tok.shape)} fp16')

    tail = VF.Tail(m, cut, len(D.CLASS_NAMES)).to(device).float()
    n_tr = sum(p.numel() for p in tail.parameters())
    cw = D.compute_class_weights(d['train']).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt = optim.AdamW([
        {'params': tail.blocks.parameters(), 'lr': lr_blocks},
        {'params': tail.ln.parameters(),     'lr': lr_blocks},
        {'params': tail.head.parameters(),   'lr': lr_head}], weight_decay=weight_decay)
    sched = optim.lr_scheduler.LambdaLR(opt, lambda e: (e + 1) / 2 if e < 1 else
            0.5 * (1 + np.cos(np.pi * (e - 1) / max(1, epochs - 1))) * 0.99 + 0.01)  # 1 warm-up epoch + cosine

    @torch.no_grad()
    def predict(tok, bs=512):
        tail.eval(); lg = []
        for i in range(0, len(tok), bs):
            lg.append(tail(tok[i:i+bs].to(device).float()).float().cpu())
        return torch.cat(lg)

    hist = {k: [] for k in ['train_loss','train_acc','train_macro_f1','val_loss','val_acc','val_macro_f1','lr','epoch_time']}
    best_f1, best_state, bad = -1.0, None, 0
    N = tr_tok.shape[1]
    start = time.time()
    for ep in range(1, epochs + 1):
        t1 = time.time(); tail.train()
        perm = torch.randperm(N); vsel = torch.randint(0, len(VIEWS), (N,))
        tot, preds, tgts = 0.0, [], []
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            xb = tr_tok[vsel[idx], idx].to(device).float(); yb = y['train'][idx].to(device)
            logits = tail(xb); loss = criterion(logits, yb)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(tail.parameters(), 1.0); opt.step()
            tot += loss.item() * len(idx)
            preds.append(logits.argmax(1).cpu()); tgts.append(yb.cpu())
        sched.step()
        p, t = torch.cat(preds).numpy(), torch.cat(tgts).numpy()
        lg = predict(va_tok); vp = lg.argmax(1).numpy(); vt = y['val'].numpy()
        vl = criterion(lg.to(device), y['val'].to(device)).item()
        vf1 = f1_score(vt, vp, average='macro', zero_division=0)
        for k, v in zip(hist, [tot/N, accuracy_score(t, p), f1_score(t, p, average='macro', zero_division=0),
                               vl, accuracy_score(vt, vp), vf1, opt.param_groups[0]['lr'], time.time()-t1]):
            hist[k].append(v)
        print(f'Epoch [{ep:02d}/{epochs}] {time.time()-t1:.1f}s | Train Loss {tot/N:.4f} | '
              f'Val Loss {vl:.4f} Acc {hist["val_acc"][-1]:.4f} F1 {vf1:.4f} (Best {max(best_f1, vf1):.4f})')
        if vf1 > best_f1:
            best_f1, bad = vf1, 0; best_state = copy.deepcopy(tail.state_dict()); best_ep = ep
        else:
            bad += 1
            if bad >= patience:
                print(f'Early stopping after {ep} epochs.'); break
    train_s = time.time() - start
    print(f'Fine-tuning done in {train_s/60:.2f} min (+ caching). Best Val Macro-F1: {best_f1:.4f}')

    pd.DataFrame(hist).to_csv(os.path.join(CURVES_DIR, f'vitb16_seed{seed}_history.csv'), index=False)
    plot_training_history(hist, 'vitb16', seed, os.path.join(CURVES_DIR, f'vitb16_seed{seed}_curves.png'))

    tail.load_state_dict(best_state)                       # m now holds the best fine-tuned weights
    torch.save({'epoch': best_ep, 'model_state_dict': m.cpu().state_dict(), 'best_val_f1': best_f1,
                'model_name': 'vitb16', 'seed': seed, 'input_hw': input_hw, 'cut': cut, 'split_version': SPLIT_MANIFEST['split_version'], 'splits_sha256': SPLIT_MANIFEST['splits_sha256']},
               os.path.join(CKPT_DIR, f'vitb16_seed{seed}_best.pt'))
    tail.to(device)
    lg = predict(te_tok); probs = torch.softmax(lg, 1).numpy(); tp = lg.argmax(1).numpy(); tt = y['test'].numpy()
    te_loss = criterion(lg.to(device), y['test'].to(device)).item()
    pcf = f1_score(tt, tp, average=None, zero_division=0)
    res = {'model': 'vitb16', 'seed': seed, 'split_version': SPLIT_MANIFEST['split_version'], 'splits_sha256': SPLIT_MANIFEST['splits_sha256'], 'parameters': n_tr, 'best_val_f1': float(best_f1),
           'test_loss': te_loss, 'test_accuracy': float(accuracy_score(tt, tp)),
           'test_macro_f1': float(f1_score(tt, tp, average='macro', zero_division=0)),
           'total_training_seconds': float(time.time() - t0),
           'per_class_f1': {D.CLASS_NAMES[i]: float(pcf[i]) for i in range(len(pcf))}}
    with open(os.path.join(METRICS_DIR, f'vitb16_seed{seed}_metrics.json'), 'w') as f: json.dump(res, f, indent=2)
    np.save(os.path.join(PREDS_DIR, f'vitb16_seed{seed}_test_preds.npy'), tp)
    np.save(os.path.join(PREDS_DIR, f'vitb16_seed{seed}_test_probs.npy'), probs)
    np.save(os.path.join(PREDS_DIR, 'test_targets.npy'), tt)
    print(f'Test Acc {res["test_accuracy"]:.4f} | Test Macro-F1 {res["test_macro_f1"]:.4f}')
    return res

---
## 4. Train Scratch Models (M1 & M2)

- **M1 (ViT-Scratch):** Image-only baseline built from scratch with patch embedding and scaled dot-product self-attention.
- **M2 (Fusion-ViT):** Multimodal architecture with learnable gated fusion between the ViT image token and the 33-dim thermal feature representation.

*Execution time on Apple Silicon (MPS): ~1.5 - 2 minutes per run.*

In [6]:
# ── Train M1: ViT-Scratch (Seed 0) ────────────────────────────────────────────
m1_res = run_training_experiment(
    model_name='vit_scratch',
    seed=0,
    epochs=35,
    patience=8,
    lr=5e-4,
    weight_decay=0.05,
    batch_size=128
)

Skipping vit_scratch (seed 0) — already trained (../outputs/results/metrics/vit_scratch_seed0_metrics.json).


In [7]:
# ── Train M2: Fusion-ViT (Proposed Method, Seed 0) ───────────────────────────
m2_res = run_training_experiment(
    model_name='fusion_vit',
    seed=0,
    epochs=35,
    patience=8,
    lr=5e-4,
    weight_decay=0.05,
    batch_size=128
)

Skipping fusion_vit (seed 0) — already trained (../outputs/results/metrics/fusion_vit_seed0_metrics.json).


---
## 5. Train Pretrained Benchmark Models (M3, B1, B2, B3)

**M3 (ViT-B/16) uses a fast partial fine-tuning pipeline** (`vit_fast.py`): input 160×96 (aspect-preserving, 61 tokens), blocks 0–7 frozen and run once with cached outputs, blocks 8–11 + head trained on the cache with layer-wise LRs. B1–B3 still use `run_training_experiment` (frozen backbone, lr 1e-3, 10 epochs).

In [8]:
# ── Train M3: ViT-B/16 (Seed 0) — fast partial fine-tuning on cached tokens ──
m3_res = run_vitb16_fast(seed=0)

Skipping vitb16 (seed 0) — already trained (../outputs/results/metrics/vitb16_seed0_metrics.json).


In [9]:
# ── Train B1: Swin-T (Seed 0) ────────────────────────────────────────────────
b1_res = run_training_experiment(
    model_name='swin_t',
    seed=0,
    epochs=10,
    patience=5,
    lr=1e-3,
    weight_decay=0.01,
    batch_size=64
)

Skipping swin_t (seed 0) — already trained (../outputs/results/metrics/swin_t_seed0_metrics.json).


In [10]:
# ── Train B2: ResNet-50 (Seed 0) ─────────────────────────────────────────────
b2_res = run_training_experiment(
    model_name='resnet50',
    seed=0,
    epochs=10,
    patience=5,
    lr=1e-3,
    weight_decay=0.01,
    batch_size=64
)

Skipping resnet50 (seed 0) — already trained (../outputs/results/metrics/resnet50_seed0_metrics.json).


In [11]:
# ── Train B3: MobileNetV3-Large (Seed 0) ──────────────────────────────────────
b3_res = run_training_experiment(
    model_name='mobilenetv3',
    seed=0,
    epochs=10,
    patience=5,
    lr=1e-3,
    weight_decay=0.01,
    batch_size=64
)

Skipping mobilenetv3 (seed 0) — already trained (../outputs/results/metrics/mobilenetv3_seed0_metrics.json).


---
## 6. Multi-Seed Training Harness (Seeds 1 & 2)

To satisfy Criterion 6 (statistical significance testing and confidence intervals), we execute seeds 1 and 2 across all models.
Set `RUN_EXTRA_SEEDS = True` when executing the full pipeline.

In [12]:
import gc

RUN_EXTRA_SEEDS = True   # Set to True to train remaining seeds (1 and 2)
SKIP_EXISTING   = True   # Skip runs whose metrics JSON already exists

if RUN_EXTRA_SEEDS:
    all_models = ['vit_scratch', 'fusion_vit', 'vitb16', 'swin_t', 'resnet50', 'mobilenetv3']
    for seed in [1, 2]:
        for m_name in all_models:
            if m_name == 'vitb16':
                run_vitb16_fast(seed=seed, skip_existing=SKIP_EXISTING)
            else:
                is_sc = m_name in ['vit_scratch', 'fusion_vit']
                run_training_experiment(
                    model_name=m_name, seed=seed,
                    epochs=35 if is_sc else 10, patience=8 if is_sc else 5,
                    lr=5e-4 if is_sc else 1e-3, weight_decay=0.05 if is_sc else 0.01,
                    batch_size=128 if is_sc else 64, skip_existing=SKIP_EXISTING)
            gc.collect()
            if torch.backends.mps.is_available():
                torch.mps.empty_cache()
            elif torch.cuda.is_available():
                torch.cuda.empty_cache()
else:
    print('Multi-seed training skipped for quick demonstration. Set RUN_EXTRA_SEEDS = True to run all seeds.')

Skipping vit_scratch (seed 1) — already trained (../outputs/results/metrics/vit_scratch_seed1_metrics.json).
Skipping fusion_vit (seed 1) — already trained (../outputs/results/metrics/fusion_vit_seed1_metrics.json).
Skipping vitb16 (seed 1) — already trained (../outputs/results/metrics/vitb16_seed1_metrics.json).
Skipping swin_t (seed 1) — already trained (../outputs/results/metrics/swin_t_seed1_metrics.json).
Skipping resnet50 (seed 1) — already trained (../outputs/results/metrics/resnet50_seed1_metrics.json).
Skipping mobilenetv3 (seed 1) — already trained (../outputs/results/metrics/mobilenetv3_seed1_metrics.json).
Skipping vit_scratch (seed 2) — already trained (../outputs/results/metrics/vit_scratch_seed2_metrics.json).
Skipping fusion_vit (seed 2) — already trained (../outputs/results/metrics/fusion_vit_seed2_metrics.json).
Skipping vitb16 (seed 2) — already trained (../outputs/results/metrics/vitb16_seed2_metrics.json).
Skipping swin_t (seed 2) — already trained (../outputs/resu

---
## 6B. Multimodal Ablation — Thermal-Feature-Only Baselines (33-d Features)

To isolate how much predictive signal comes from the 33 engineered thermal features alone (since they are derived from the same `40×24` infrared pixels rather than an independent hardware sensor), we evaluate:
1. **`FeatureMLP` (Seeds 0–2):** Matches `FusionViT`'s exact `33 → 128 → 192 → 12` feature branch and training recipe (AdamW, cosine annealing, inverse-sqrt class weights, label smoothing `0.1`, early stopping on Val Macro-F1).
2. **`LogisticRegression` & `HistGradientBoostingClassifier`:** Classical tabular baselines fit strictly on train-split-standardised 33-d features.

Outputs are saved to `outputs/results/ablation/` without overwriting any of the 6 core deep models in `outputs/results/metrics/`.

In [13]:
# ── Thermal-feature-only ablation (FeatureMLP 3 seeds + classical baselines) ─
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

ABL_DIR = os.path.join(RESULTS_DIR, 'ablation')
os.makedirs(ABL_DIR, exist_ok=True)

df_sp = pd.read_csv('../outputs/splits.csv')
feats_all = np.load('../outputs/features_33d.npy')
tr_m = (df_sp['split'] == 'train').values
va_m = (df_sp['split'] == 'val').values
te_m = (df_sp['split'] == 'test').values

f_mean = feats_all[tr_m].mean(axis=0).astype(np.float32)
f_std  = np.where(feats_all[tr_m].std(axis=0) < 1e-6, 1.0, feats_all[tr_m].std(axis=0)).astype(np.float32)

X_tr = (feats_all[tr_m] - f_mean) / f_std
X_va = (feats_all[va_m] - f_mean) / f_std
X_te = (feats_all[te_m] - f_mean) / f_std
y_tr = df_sp.loc[tr_m, 'label'].to_numpy()
y_va = df_sp.loc[va_m, 'label'].to_numpy()
y_te = df_sp.loc[te_m, 'label'].to_numpy()

cw_vec = D.compute_class_weights(df_sp[tr_m]).to(DEVICE)

def run_feature_mlp(seed: int, epochs: int = 35, patience: int = 8, lr: float = 5e-4, weight_decay: float = 0.05):
    mpath = os.path.join(ABL_DIR, f'feature_mlp_seed{seed}_metrics.json')
    if os.path.exists(mpath):
        with open(mpath) as f:
            return json.load(f)
    set_seed(seed)
    t0 = time.time()
    model = M.FeatureMLP(feat_dim=33, embed_dim=192, num_classes=len(D.CLASS_NAMES)).to(DEVICE)
    n_par = M.count_parameters(model)
    crit = nn.CrossEntropyLoss(weight=cw_vec, label_smoothing=0.1)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)

    Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
    ytr_t = torch.tensor(y_tr, dtype=torch.long)
    Xva_t = torch.tensor(X_va, dtype=torch.float32, device=DEVICE)
    Xte_t = torch.tensor(X_te, dtype=torch.float32, device=DEVICE)

    best_f1, best_state, bad = -1.0, None, 0
    bs = 128
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(Xtr_t), bs):
            idx = perm[i:i+bs]
            xb, yb = Xtr_t[idx].to(DEVICE), ytr_t[idx].to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = model(Xva_t).argmax(1).cpu().numpy()
        vf1 = f1_score(y_va, vp, average='macro', zero_division=0)
        if vf1 > best_f1:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        te_probs = torch.softmax(model(Xte_t), dim=1).cpu().numpy()
    te_preds = te_probs.argmax(axis=1)
    pcf = f1_score(y_te, te_preds, average=None, zero_division=0)
    res = {
        'model': 'feature_mlp', 'seed': seed, 'parameters': int(n_par),
        'best_val_f1': float(best_f1),
        'test_accuracy': float(accuracy_score(y_te, te_preds)),
        'test_macro_f1': float(f1_score(y_te, te_preds, average='macro', zero_division=0)),
        'total_training_seconds': float(time.time() - t0),
        'per_class_f1': {D.CLASS_NAMES[i]: float(pcf[i]) for i in range(len(pcf))},
    }
    with open(mpath, 'w') as f:
        json.dump(res, f, indent=2)
    np.save(os.path.join(ABL_DIR, f'feature_mlp_seed{seed}_test_preds.npy'), te_preds)
    np.save(os.path.join(ABL_DIR, f'feature_mlp_seed{seed}_test_probs.npy'), te_probs)
    return res


mlp_res = [run_feature_mlp(s) for s in [0, 1, 2]]

# Classical feature-only baselines
lr_path = os.path.join(ABL_DIR, 'logreg_feat_metrics.json')
if not os.path.exists(lr_path):
    t0 = time.time()
    clf_lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    clf_lr.fit(X_tr, y_tr)
    p_lr = clf_lr.predict(X_te)
    pr_lr = clf_lr.predict_proba(X_te)
    pcf = f1_score(y_te, p_lr, average=None, zero_division=0)
    lr_res = {
        'model': 'logreg_feat', 'seed': 42, 'parameters': int(clf_lr.coef_.size + clf_lr.intercept_.size),
        'best_val_f1': float(f1_score(y_va, clf_lr.predict(X_va), average='macro', zero_division=0)),
        'test_accuracy': float(accuracy_score(y_te, p_lr)),
        'test_macro_f1': float(f1_score(y_te, p_lr, average='macro', zero_division=0)),
        'total_training_seconds': float(time.time() - t0),
        'per_class_f1': {D.CLASS_NAMES[i]: float(pcf[i]) for i in range(len(pcf))},
    }
    with open(lr_path, 'w') as f: json.dump(lr_res, f, indent=2)
    np.save(os.path.join(ABL_DIR, 'logreg_feat_test_preds.npy'), p_lr)
    np.save(os.path.join(ABL_DIR, 'logreg_feat_test_probs.npy'), pr_lr)
else:
    with open(lr_path) as f: lr_res = json.load(f)

hg_path = os.path.join(ABL_DIR, 'histgb_feat_metrics.json')
if not os.path.exists(hg_path):
    t0 = time.time()
    clf_hg = HistGradientBoostingClassifier(class_weight='balanced', random_state=42, max_iter=200)
    clf_hg.fit(X_tr, y_tr)
    p_hg = clf_hg.predict(X_te)
    pr_hg = clf_hg.predict_proba(X_te)
    pcf = f1_score(y_te, p_hg, average=None, zero_division=0)
    hg_res = {
        'model': 'histgb_feat', 'seed': 42, 'parameters': 0,
        'best_val_f1': float(f1_score(y_va, clf_hg.predict(X_va), average='macro', zero_division=0)),
        'test_accuracy': float(accuracy_score(y_te, p_hg)),
        'test_macro_f1': float(f1_score(y_te, p_hg, average='macro', zero_division=0)),
        'total_training_seconds': float(time.time() - t0),
        'per_class_f1': {D.CLASS_NAMES[i]: float(pcf[i]) for i in range(len(pcf))},
    }
    with open(hg_path, 'w') as f: json.dump(hg_res, f, indent=2)
    np.save(os.path.join(ABL_DIR, 'histgb_feat_test_preds.npy'), p_hg)
    np.save(os.path.join(ABL_DIR, 'histgb_feat_test_probs.npy'), pr_hg)
else:
    with open(hg_path) as f: hg_res = json.load(f)

# Compile ablation comparison table (Feature-only vs Image-only M1 vs Multimodal M2)
def _load_seed_stats(m_key):
    runs = []
    for s in [0, 1, 2]:
        with open(os.path.join(METRICS_DIR, f'{m_key}_seed{s}_metrics.json')) as f:
            runs.append(json.load(f))
    return runs

m1_runs = _load_seed_stats('vit_scratch')
m2_runs = _load_seed_stats('fusion_vit')

abl_rows = [
    {'Variant': 'Feature-Only LogReg (33-d)', 'Family': 'Classical (Linear)', 'Modality': '33-d features', 'Seeds': 1,
     'Neural Trainable Params': '—', 'Model Complexity / Size': f"{lr_res['parameters']} linear weights (33×12+12)",
     'Test Acc Mean': lr_res['test_accuracy'], 'Test Acc Std': 0.0,
     'Macro-F1 Mean': lr_res['test_macro_f1'], 'Macro-F1 Std': 0.0},
    {'Variant': 'Feature-Only HistGB (33-d)', 'Family': 'Classical (GBDT)', 'Modality': '33-d features', 'Seeds': 1,
     'Neural Trainable Params': '—', 'Model Complexity / Size': '200 boosting iterations × 12 classes',
     'Test Acc Mean': hg_res['test_accuracy'], 'Test Acc Std': 0.0,
     'Macro-F1 Mean': hg_res['test_macro_f1'], 'Macro-F1 Std': 0.0},
    {'Variant': 'Feature-Only MLP (33-d)', 'Family': 'Neural (MLP)', 'Modality': '33-d features', 'Seeds': 3,
     'Neural Trainable Params': f"{mlp_res[0]['parameters']:,}", 'Model Complexity / Size': f"{mlp_res[0]['parameters']:,} params (33→128→192→12)",
     'Test Acc Mean': float(np.mean([r['test_accuracy'] for r in mlp_res])),
     'Test Acc Std':  float(np.std([r['test_accuracy'] for r in mlp_res], ddof=1)),
     'Macro-F1 Mean': float(np.mean([r['test_macro_f1'] for r in mlp_res])),
     'Macro-F1 Std':  float(np.std([r['test_macro_f1'] for r in mlp_res], ddof=1))},
    {'Variant': 'Image-Only ViT-Scratch (M1)', 'Family': 'Neural (ViT)', 'Modality': '40×24 image', 'Seeds': 3,
     'Neural Trainable Params': f"{m1_runs[0]['parameters']:,}", 'Model Complexity / Size': f"{m1_runs[0]['parameters']:,} params (6L, d=192, 3H)",
     'Test Acc Mean': float(np.mean([r['test_accuracy'] for r in m1_runs])),
     'Test Acc Std':  float(np.std([r['test_accuracy'] for r in m1_runs], ddof=1)),
     'Macro-F1 Mean': float(np.mean([r['test_macro_f1'] for r in m1_runs])),
     'Macro-F1 Std':  float(np.std([r['test_macro_f1'] for r in m1_runs], ddof=1))},
    {'Variant': 'Gated Fusion-ViT (M2)', 'Family': 'Neural (Multimodal)', 'Modality': 'Image + 33-d gated', 'Seeds': 3,
     'Neural Trainable Params': f"{m2_runs[0]['parameters']:,}", 'Model Complexity / Size': f"{m2_runs[0]['parameters']:,} params (ViT + MLP + Gate)",
     'Test Acc Mean': float(np.mean([r['test_accuracy'] for r in m2_runs])),
     'Test Acc Std':  float(np.std([r['test_accuracy'] for r in m2_runs], ddof=1)),
     'Macro-F1 Mean': float(np.mean([r['test_macro_f1'] for r in m2_runs])),
     'Macro-F1 Std':  float(np.std([r['test_macro_f1'] for r in m2_runs], ddof=1))},
]
abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv(os.path.join(ABL_DIR, 'ablation_results.csv'), index=False)
print('=== Modality Ablation Summary ===')
print(abl_df.to_string(index=False))

=== Modality Ablation Summary ===
                    Variant              Family           Modality  Seeds Neural Trainable Params              Model Complexity / Size  Test Acc Mean  Test Acc Std  Macro-F1 Mean  Macro-F1 Std
 Feature-Only LogReg (33-d)  Classical (Linear)      33-d features      1                       —        408 linear weights (33×12+12)       0.492500      0.000000       0.376205      0.000000
 Feature-Only HistGB (33-d)    Classical (GBDT)      33-d features      1                       — 200 boosting iterations × 12 classes       0.641000      0.000000       0.488289      0.000000
    Feature-Only MLP (33-d)        Neural (MLP)      33-d features      3                  31,436        31,436 params (33→128→192→12)       0.703000      0.004770       0.503709      0.005129
Image-Only ViT-Scratch (M1)        Neural (ViT)        40×24 image      3               2,683,596     2,683,596 params (6L, d=192, 3H)       0.710167      0.008963       0.512087      0.015353
 

---
## 7. Multi-Seed Comparative Summary (Seeds 0–2)

Aggregates all saved per-seed metrics JSONs from `outputs/results/metrics/` into both a per-run table and a Mean ± Std summary across seeds, clearly separating **Total Parameters** from **Trainable Parameters**.

**Experimental Protocol & Scope Notes:**
- **Benchmark configuration differences:** `vitb16` (M3) uses aspect-preserving $160\times 96$ input with blocks 8–11 + head fine-tuned (`28.36M` trainable of `85.70M` total), whereas `swin_t`, `resnet50`, and `mobilenetv3` (B1–B3) are frozen-backbone linear probes at $224\times 224$ (`9k–25k` trainable params). `M1`/`M2` train end-to-end from scratch at native $40\times 24$.
- **Task scope:** All models perform **12-class image classification** (not bounding-box object detection, as `InfraredSolarModules` provides image-level labels only).

In [14]:
# ── Aggregate and print multi-seed test results (Total vs Trainable Params) ─
TOTAL_PARAMS_MAP = {
    'vit_scratch': sum(p.numel() for p in M.get_model('vit_scratch').parameters()),
    'fusion_vit':  sum(p.numel() for p in M.get_model('fusion_vit').parameters()),
    'vitb16':      sum(p.numel() for p in VF.build_vitb16((160, 96), pretrained=False, num_classes=len(D.CLASS_NAMES)).parameters()),
    'swin_t':      sum(p.numel() for p in M.get_model('swin_t').parameters()),
    'resnet50':    sum(p.numel() for p in M.get_model('resnet50').parameters()),
    'mobilenetv3': sum(p.numel() for p in M.get_model('mobilenetv3').parameters()),
}

per_run_rows = []
for f_name in sorted(os.listdir(METRICS_DIR)):
    if f_name.endswith('_metrics.json'):
        with open(os.path.join(METRICS_DIR, f_name)) as f:
            data = json.load(f)
        m_key = data['model']
        per_run_rows.append({
            'Model': m_key,
            'Seed': int(data['seed']),
            'Total Params (M)': TOTAL_PARAMS_MAP[m_key] / 1e6,
            'Trainable Params (M)': data['parameters'] / 1e6,
            'Val Macro-F1': float(data['best_val_f1']),
            'Test Macro-F1': float(data['test_macro_f1']),
            'Test Accuracy': float(data['test_accuracy']),
            'Train Time (min)': float(data['total_training_seconds']) / 60.0,
        })

if per_run_rows:
    runs_df = pd.DataFrame(per_run_rows).sort_values(['Model', 'Seed']).reset_index(drop=True)
    runs_df.to_csv(os.path.join(METRICS_DIR, 'multiseed_runs.csv'), index=False)

    agg_rows = []
    for m_name, grp in runs_df.groupby('Model', sort=False):
        agg_rows.append({
            'Model': m_name,
            'Seeds': len(grp),
            'Total Params': f"{grp['Total Params (M)'].iloc[0]:.2f}M",
            'Trainable Params': f"{grp['Trainable Params (M)'].iloc[0]:.2f}M",
            'Val Macro-F1': f"{grp['Val Macro-F1'].mean():.4f} ± {grp['Val Macro-F1'].std(ddof=1):.4f}",
            'Test Macro-F1': f"{grp['Test Macro-F1'].mean():.4f} ± {grp['Test Macro-F1'].std(ddof=1):.4f}",
            'Test Accuracy': f"{grp['Test Accuracy'].mean():.4f} ± {grp['Test Accuracy'].std(ddof=1):.4f}",
            'Train Time (min)': f"{grp['Train Time (min)'].mean():.1f} ± {grp['Train Time (min)'].std(ddof=1):.1f}",
            '_sort_f1': grp['Test Macro-F1'].mean(),
        })
    summary_df = pd.DataFrame(agg_rows).sort_values('_sort_f1', ascending=False).drop(columns=['_sort_f1']).reset_index(drop=True)
    summary_df.to_csv(os.path.join(METRICS_DIR, 'multiseed_summary.csv'), index=False)

    print('\nMulti-Seed Benchmark Summary (Mean ± Std across seeds):')
    print(summary_df.to_string(index=False))
else:
    print('No metrics found yet. Run the training cells above.')

# Write checkpoint & metrics provenance manifest (Priority 5)
prov_rows = []
for f_name in sorted(os.listdir(METRICS_DIR)):
    if f_name.endswith('_metrics.json'):
        with open(os.path.join(METRICS_DIR, f_name)) as f:
            d = json.load(f)
        m_key, sd = d['model'], int(d['seed'])
        ckpt_p = os.path.join(CKPT_DIR, f'{m_key}_seed{sd}_best.pt')
        prov_rows.append({
            'model': m_key, 'seed': sd,
            'split_version': d.get('split_version', SPLIT_MANIFEST['split_version']),
            'splits_sha256': d.get('splits_sha256', SPLIT_MANIFEST['splits_sha256']),
            'total_params': TOTAL_PARAMS_MAP[m_key],
            'trainable_params': int(d['parameters']),
            'checkpoint_file': f'{m_key}_seed{sd}_best.pt',
            'checkpoint_sha256': D.compute_file_sha256(ckpt_p) if os.path.exists(ckpt_p) else 'missing',
            'checkpoint_mb': round(os.path.getsize(ckpt_p) / (1024**2), 2) if os.path.exists(ckpt_p) else 0.0,
            'test_accuracy': float(d['test_accuracy']),
            'test_macro_f1': float(d['test_macro_f1']),
        })
prov_df = pd.DataFrame(prov_rows).sort_values(['model', 'seed']).reset_index(drop=True)
prov_df.to_csv(os.path.join(RESULTS_DIR, 'provenance_manifest.csv'), index=False)
print(f'Saved checkpoint provenance manifest → {os.path.join(RESULTS_DIR, "provenance_manifest.csv")} ({len(prov_df)} runs)')


Multi-Seed Benchmark Summary (Mean ± Std across seeds):
      Model  Seeds Total Params Trainable Params    Val Macro-F1   Test Macro-F1   Test Accuracy Train Time (min)
     vitb16      3       85.70M           28.36M 0.6366 ± 0.0146 0.5971 ± 0.0126 0.7972 ± 0.0008      66.9 ± 29.4
 fusion_vit      3        2.79M            2.79M 0.5322 ± 0.0082 0.5207 ± 0.0123 0.7038 ± 0.0058       21.6 ± 6.9
vit_scratch      3        2.68M            2.68M 0.5539 ± 0.0171 0.5121 ± 0.0154 0.7102 ± 0.0090       20.7 ± 7.9
     swin_t      3       27.53M            0.01M 0.4365 ± 0.0061 0.4118 ± 0.0058 0.6615 ± 0.0030       43.2 ± 4.2
   resnet50      3       23.53M            0.02M 0.3804 ± 0.0082 0.3972 ± 0.0076 0.6103 ± 0.0088       33.2 ± 2.8
mobilenetv3      3        4.22M            0.02M 0.3911 ± 0.0054 0.3858 ± 0.0023 0.6047 ± 0.0025        9.9 ± 0.7
Saved checkpoint provenance manifest → ../outputs/results/provenance_manifest.csv (18 runs)
